# Veon Scientific i.prep 2

```{device-card} veon-iprep2
```

| Property | Value |
|---|---|
| Channels | reported by the instrument; 8 on the machine this was written against |
| Connection | HTTP and WebSocket over the network |
| Deck | six zones, each taking an SBS footprint on its long edge |
| Units | millimetres, microlitres, microlitres per second |

**This integration has not been verified on physical hardware.** It has been driven against a real
i.prep 2 service, but not against one moving liquid. Review what it does before pointing it at an
instrument, and please report back once it has been checked:
[issue #1218](https://github.com/PyLabRobot/pylabrobot/issues/1218).

This guide covers what is here today: connecting, reading what the instrument is, and laying out
its deck. Moving the instrument and handling liquid come later.

## How it talks

The i.prep 2 runs its own service and answers over HTTP, so PyLabRobot is a client of it rather
than a driver of its firmware. There is nothing to frame and no units to convert: the instrument
already speaks millimetres and microlitres.

Two connections are opened. Requests go over HTTP, and each one is answered when the operation it
asked for has *finished* rather than when it was accepted - so a long operation holds its request
open for as long as it runs, and a second command meanwhile is refused rather than queued.
Alongside that, the instrument pushes a stream of events over a WebSocket, which PyLabRobot
follows so its model of the machine keeps up without polling.

## Before you start

Put the instrument and this computer on the same network and find the instrument's address. An
i.prep 2 on a bench is unauthenticated, which is the usual case and what the example below
assumes.

One reachable over a public address needs a token, which goes in as `api_key` and requires
`secure=True`: PyLabRobot refuses to send a credential over an unencrypted connection, and keeps
it out of its logs, its captures and anything it serializes.

## Setup

Connecting reads the instrument: what it is, what it has, and how it was left. Nothing moves.

In [ ]:
from pylabrobot.veon import IPrep2

iprep2 = IPrep2(host="iprep2.local")  # add api_key=..., secure=True for an instrument that needs one
await iprep2.setup()
iprep2

## What the instrument is

Identity is what the machine calls itself, rather than what PyLabRobot calls the resource. It also
says outright whether it is simulated, so you never have to infer that from the address you used.

In [ ]:
identity = iprep2.identity
print(identity.name, identity.model, identity.serial)
print("simulated:", identity.simulated)
print("versions:", identity.versions)

## What the instrument has

Read from the instrument rather than assumed, because two machines of the same model need not
agree: channel count, axis travel, the zones on the deck and the liquid classes installed all come
off the instrument at setup.

Channel numbers here are the instrument's, which start at 1. PyLabRobot indexes channels from 0,
and `driver.channel_number` / `driver.channel_index` convert between the two.

In [ ]:
capabilities = iprep2.capabilities
print("channels:", capabilities.pipette.channels, "at", capabilities.pipette.channel_pitch, "mm")
print("max volume:", capabilities.pipette.max_volume, "uL")
print("axes:", capabilities.motion.axes)
print("x travel:", capabilities.motion.travel["x"].as_tuple())
print("zones:", capabilities.deck.zones)
print("liquid classes:", capabilities.liquid_classes)

## Whether it is free

Cheap enough to ask before dispatching work, and the way to find out without either moving the
instrument or provoking a rejection to read the refusal off.

`left_dirty` is the case worth watching for: the instrument is free, but a previous run finished
with tips still on and the head somewhere no protocol chose.

In [ ]:
readiness = await iprep2.readiness()
print("busy:", readiness.busy, "owner:", readiness.owner)
print("at home:", readiness.at_home, "axes away:", readiness.axes_away_from_home)
print("tips on channels:", readiness.tips_attached)
print("left dirty:", readiness.left_dirty)

## The deck

The deck has six zones, and labware goes into one by name. The zones are where PyLabRobot's
description of this deck puts them, moved by whatever calibration the instrument reported at
setup - so the model reflects where *this* machine found each zone, not only where the drawing
says it is.

In [ ]:
print(iprep2.deck.zone_names)
print(iprep2.deck.summary())

## Putting labware on it

This deck holds an SBS footprint **on its long edge**, and PyLabRobot draws a plate on its short
edge. So a plate goes in a quarter turn from how it is defined, and one handed over untouched is
refused rather than quietly rotated - rotating labware moves every well in it, and doing that
silently is how a protocol comes to aspirate from the wrong one.

`rotated(z=90)` returns a turned copy. The zone works out where the turned plate sits, so its
wells land where the instrument expects them.

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

plate = Cor_96_wellplate_360ul_Fb(name="source_plate")
iprep2.deck.assign_child_at_zone(plate.rotated(z=90), "Zone2")
print(iprep2.deck.summary())

Everything on the deck is a descendant of the instrument carrying it, so a well's position is
known in the instrument's own coordinates without anything further being worked out.

In [ ]:
placed = iprep2.deck.zones["Zone2"]
print("plate origin:", placed.get_absolute_location())
print("well A1:", placed.get_item("A1").get_absolute_location())
print("rooted at the instrument:", placed.get_root() is iprep2)

## Following what the instrument does

The instrument pushes an event whenever something happens on it - a move starting and finishing, a
tip picked up or ejected, a channel's volume changing. PyLabRobot puts each one on its event bus
under `iprep2.<name>`, so a subscriber can tell an event the instrument reported from one
PyLabRobot raised itself.

Subscribe before `setup()` to catch everything from the start.

In [ ]:
from pylabrobot.events import EventBus, set_default_event_bus

bus = EventBus()
bus.subscribe(lambda event: print(event.name, event.data.get("payload")))
set_default_event_bus(bus)

## Finishing

Stopping closes both connections and leaves the instrument exactly as it is. Nothing is homed and
no tip is ejected: what is safe to do with a tip depends on what is underneath it, and that is a
decision for whoever can see the deck.

In [ ]:
await iprep2.stop()